In [ ]:
#| default_exp preprocessing

# Preprocessing
> Utilities for processing data before training/analysis

In [ ]:
#| export
from typing import Union

import numpy as np
import pandas as pd

In [ ]:
#| hide
from nbdev import show_doc

from utilsforecast.data import generate_series

In [ ]:
#| exporti
def _determine_bound(bound, freq, times_by_id, agg) -> np.ndarray:
    if bound == 'per_serie':
        out = times_by_id[agg].values
    else:
        # the following return a scalar
        if bound == 'global':
            val = getattr(times_by_id[agg].values, agg)()
            if isinstance(freq, str):
                val = np.datetime64(val)
        else:
            if isinstance(freq, str):
                # this raises a nice error message if it isn't a valid datetime
                val = np.datetime64(bound)
            else:
                val = bound
        out = np.full(times_by_id.shape[0], val)
    if isinstance(freq, str):
        out = out.astype(f'datetime64[{freq}]')
    return out

In [ ]:
#| export
def fill_gaps(
    df: pd.DataFrame,
    freq: Union[str, int],
    start: str = 'per_serie',
    end: str = 'global',
    id_col: str = 'unique_id',
    time_col: str = 'ds',
) -> pd.DataFrame:
    """Enforce start and end datetimes for dataframe.

    Parameters
    ----------
    df : pandas DataFrame
        Input data
    freq : str or int
        Series' frequency
    start : str
        Initial timestamp for the series.
            * 'per_serie' uses each serie's first timestamp
            * 'global' uses the first timestamp seen in the data
            * Can also be a specific timestamp, e.g. '2000-01-01'
    end : str
        Initial timestamp for the series.
            * 'per_serie' uses each serie's last timestamp
            * 'global' uses the last timestamp seen in the data
            * Can also be a specific timestamp, e.g. '2000-01-01'
    id_col : str (default='unique_id')
        Column that identifies each serie.
    time_col : str (default='ds')
        Column that identifies each timestamp.
    target_col : str (default='y')
        Column that contains the target.

    Returns
    -------
    filled_df : pandas DataFrame
        Dataframe with gaps filled.
    """
    delta = np.timedelta64(1, freq) if isinstance(freq, str) else freq
    times_by_id = df.groupby(id_col)[time_col].agg(['min', 'max'])
    starts = _determine_bound(start, freq, times_by_id, 'min')
    ends = _determine_bound(end, freq, times_by_id, 'max') + delta
    sizes = ((ends - starts) / delta).astype(np.int64)
    times = np.concatenate(
        [np.arange(start, end, delta) for start, end in zip(starts, ends)]
    )
    if isinstance(freq, str):
        times = times.astype('datetime64[ns]', copy=False)
    uids = np.repeat(times_by_id.index, sizes)
    idx = pd.MultiIndex.from_arrays([uids, times], names=[id_col, time_col])
    return df.set_index([id_col, time_col]).reindex(idx).reset_index()

In [ ]:
show_doc(fill_gaps)

In [ ]:
df = pd.DataFrame(
    {
        'unique_id': [0, 0, 0, 1, 1],
        'ds': pd.to_datetime(['2020', '2021', '2023', '2021', '2022']),
        'y': np.arange(5),
    }
)
df

The default functionality is taking the current starts and only extending the end date to be the same for all series.

In [ ]:
fill_gaps(
    df,
    freq='Y',
)

We can also specify `end='per_serie'` to only fill possible gaps within each serie.

In [ ]:
fill_gaps(
    df,
    freq='Y',
    end='per_serie',
)

We can also specify an end date in the future.

In [ ]:
fill_gaps(
    df,
    freq='Y',
    end='2024',
)

We can set all series to start at the same time.

In [ ]:
fill_gaps(
    df,
    freq='Y',
    start='global'
)

We can also set a common start date for all series (which can be earlier than their current starts).

In [ ]:
fill_gaps(
    df,
    freq='Y',
    start='2019',
)

In case the times are integers the frequency, start and end must also be integers.

In [ ]:
df = pd.DataFrame(
    {
        'unique_id': [0, 0, 0, 1, 1],
        'ds': [2020, 2021, 2023, 2021, 2022],
        'y': np.arange(5),
    }
)
df

In [ ]:
fill_gaps(
    df,
    freq=1,
    start=2019,
    end=2024,
)

In [ ]:
#| hide
n_series = 1_000
series = generate_series(n_series, equal_ends=True)

# keep the start date of each serie
first_dates = series.groupby('unique_id').head(1)
uneven = series.drop(first_dates.index).sample(int(0.5 * series.shape[0]), random_state=0)
uneven = pd.concat([first_dates, uneven])

end = series['ds'].max()
actual = fill_gaps(uneven, freq='D', end=end)
assert uneven.shape[0] < series.shape[0]
assert actual.shape[0] == series.shape[0]
pd.testing.assert_frame_equal(
    actual.groupby('unique_id')['ds'].agg(['min', 'max']),
    series.groupby('unique_id')['ds'].agg(['min', 'max']),
)